In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

import importlib
import model.model
import model.grid
import model
importlib.reload(model.model)
importlib.reload(model.utils)
importlib.reload(model.grid)

from model.model import QGM
from model.utils import Solver 
from model.grid import Grid

jax.config.update("jax_platform_name", "cpu")

# === Parameter definitions === #
master_key = jax.random.PRNGKey(0)
master_key, key_init, init1, init2 = jax.random.split(master_key, num=4)

params = {
        'nx': 256, 
        'Lx': 2*jnp.pi, 
        'beta': 10, 
        'k_beta':1.0, #for now we're just doing this for the forcing range idgaf
        'eta': 5, 
        'mu': 0.0,
        'nu': 1e-8,
        'dt': 1e-2,
        'k_f': 16.0,        # central forcing wavenumber
        'k_width': 1.5,    # width of the ring
        'epsilon': 1e-3,   #energy input 
        'key':key_init
}

# === Set up model === 
model = QGM(params) 
model.initialize()

In [ ]:
import jax
import jax.numpy as jnp
from jax.numpy.fft import rfftn, irfftn
import numpy as np

def kinetic_energy_from_qh(qh, grid):
 # KE = 0.5 * sum(|qh|^2 * invK2) (spectral formula)
 ke_spec = 0.5 * jnp.sum(jnp.abs(qh)**2 * grid.invK2)
 return float(ke_spec)

def diagnostics_one_step(model):
 grid = model.grid
 # spectral vorticity
 qh = rfftn(model.fields['zeta'], axes=(-2, -1))
 ke = kinetic_energy_from_qh(qh, grid)
 max_zeta = float(jnp.max(jnp.abs(model.fields['zeta'])))
 # compute forcing in same way as rhs (uses model.forcing_spectrum and model._key)
 key = getattr(model, '_key', model.parameters.get('key', jax.random.PRNGKey(0)))
 # draw a deterministic sample for diagnostics (do not change model._key)
 key_d, _ = jax.random.split(key)
 noise_phys = jax.random.normal(key_d, (grid.ny, grid.nx))
 forcing_h = rfftn(noise_phys, axes=(-2, -1)) * model.forcing_spectrum
 forcing_max = float(jnp.max(jnp.abs(forcing_h)))
 forcing_L2 = float(jnp.sqrt(jnp.sum(jnp.abs(forcing_h)**2)))
 return {
 'KE': ke,
 'max_zeta': max_zeta,
 'forcing_max': forcing_max,
 'forcing_L2': forcing_L2
 }

model 

In [ ]:
model.initialize()
nsteps = 16
print('step, KE, max|zeta|, forcing_max, forcing_L2')
for i in range(nsteps):
    diag = diagnostics_one_step(model)
    print(f"{i:2d}, {diag['KE']:.6e}, {diag['max_zeta']:.6e}, {diag['forcing_max']:.6e}, {diag['forcing_L2']:.6e}")
    try:
        model.step()
    except Exception as e:
        print('Exception at step', i, e)
        break
    # check for NaNs after the step
    z = jnp.asarray(model.fields['zeta'])
    if jnp.isnan(z).any():
        print('NaN detected after step', i+1)
        break